# Define proton conducting cavity

Use `hollow` to delineate the pathway between NHE6 (A,B:D260 = D292' and the allosteric pH sensor site C,D:His207 = His207').

(Primed residue numbers refer to the AF3 model, unprimed are canonical uniprot sequence numbers as used in the paper.)

## packages

In [15]:
import warnings
import pathlib

import numpy as np
import matplotlib.pyplot as plt

import MDAnalysis as mda

## data

We use the model with all poorly modelled stretches removed:

In [16]:
model = pathlib.Path("./NHE6_TBC1D5_GTP_MG_noloops18plus_score40plus.pdb")

In [86]:
u = mda.Universe(model)
protein = u.select_atoms("protein")

## reorient model
The membrane domains are not aligned with the (putative) membrane normal.

Try to rotate so that z axis is pararallel to membrane.

Start by ensuring we have the original coordinates from the file:

In [94]:
u.trajectory.rewind()

Only focus on the membrane protein part, NHE6:

In [95]:
nhe6 = protein.select_atoms("chainID A B")

In [96]:
ev = nhe6.principal_axes()
ev

array([[-0.17194898, -0.72210717,  0.67007074],
       [-0.62326854,  0.606497  ,  0.4936575 ],
       [-0.76286951, -0.33275011, -0.55435321]])

In [97]:
membrane_normal = np.array([0, 0, 1])
protein_vector = ev[1]  # trial and error ... 1 looks good but is "inside out"
protein_vector *= -1  # so flip manually
axis = mda.lib.transformations.rotaxis(protein_vector, membrane_normal)
angle = np.rad2deg(np.arccos(
    membrane_normal @ protein_vector / 
    (np.linalg.norm(membrane_normal) * np.linalg.norm(protein_vector))))
print(f"rotation axis = {axis} with angle {angle} degrees")

rotation axis = [-0.69739841 -0.71668365  0.        ] with angle 119.58126299815082 degrees


In [98]:
u.atoms.rotateby(angle, axis, point=nhe6.center_of_geometry())

<AtomGroup with 21918 atoms>

Center in a box


In [99]:
u.atoms.translate(-u.atoms.center_of_geometry())
box = u.atoms.bbox()
Lx, Ly, Lz = 1.05 * (box[1] - box[0])
u.trajectory.ts.dimensions = [Lx, Ly, Lz, 90., 90., 90.]

Write file to PDB (note: rerun from top if you want to change anything here because the positions are changed in place)

In [100]:
u.atoms.write("rotated.pdb")

## Key residues

* H+ binding site in NHE6: A,B:D260 = D292'
* putative pH sensor in the Rab7 GTPAse TBC1D5: H207'

In [101]:
nhe6_D260 = protein.select_atoms("chainID A B and resid 292")
print(nhe6_D260.residues)

<ResidueGroup [<Residue ASP, 292>, <Residue ASP, 292>]>


In [102]:
rab7_H207 = protein.select_atoms("chainID C D and resid 207")
print(rab7_H207.residues)

<ResidueGroup [<Residue HIS, 207>, <Residue HIS, 207>]>


## Volume of interest

**Use re-oriented model!**

Define the volume in which the pathways are located:

In [103]:
key_residues = nhe6_D260 + rab7_H207
ca_key = key_residues.select_atoms("name CA")

In [104]:
bbox = key_residues.bbox()
bbox

array([[-49.275047, -23.634737, -32.318233],
       [ 49.350334,  23.937597,  28.401197]], dtype=float32)

#### coordbrick constraints

Add some buffer towards bulk:

In [122]:
bbox[1] += [0, 0, 10]

array([49.35033417, 23.93759727, 38.40119743])

We can use these directly with the **coordbrick** constraints:
```python
 {
    'type': 'coordbrick',	
    'remove_asa_shell': False,   # True or False

    # lower left front corner of the orthorhombic box
    'coord1': (-49.275047, -23.634737, -32.318233),
                                
    # upper right back corner
    'coord2': (49.35033417, 23.93759727, 38.40119743),
 }   
```
and run `hollow`
```
hollow -g 1.0 -c constraint.txt -o hollow.pdb ../model_processing/rotated.pdb
```

#### brick (atom based)

For hollow, we can also use **brick** constraints, which are relative to residues.

We would need lower left and upper right corner of the bounding box, relative to two residues together with offsets:

Which residues are closest to corners?

In [ ]:
bbox = key_residues.bbox()
bbox

In [105]:
d = mda.lib.distances.distance_array(ca_key.positions, bbox)
d

array([[ 49.12928355,  89.90039518],
       [ 67.85312727,  76.64446518],
       [ 65.48746865, 100.34100839],
       [115.96592637,  29.6121756 ]])

In [106]:
corners = d.argmin(axis=0)

In [114]:
corners

array([0, 3])

In [107]:
lowerleft, upperright = ca_key[corners]

In [108]:
lowerleft, upperright

(<Atom 2148: CA of type C of resname ASP, resid 292 and segid A and altLoc >,
 <Atom 15465: CA of type C of resname HIS, resid 207 and segid D and altLoc >)

Find closest atoms in *model* to corners:

In [118]:
d_all = mda.lib.distances.distance_array(u.atoms.positions, bbox)
corners_all = d_all.argmin(axis=0)
lowerleft, upperright = u.atoms[corners_all]
lowerleft, upperright

(<Atom 4840: NH2 of type N of resname ARG, resid 22 and segid B and altLoc >,
 <Atom 21213: O of type O of resname ASP, resid 128 and segid F and altLoc >)

In [120]:
d_all, corners_all, d_all[corners_all]

(array([[ 93.76071467,  56.44422958],
        [ 92.29740041,  57.08204098],
        [ 91.79071648,  57.96185197],
        ...,
        [117.48059533,  10.27425839],
        [ 57.57159534,  95.65499459],
        [109.29388192,  21.11678207]], shape=(21918, 2)),
 array([ 4839, 21212]),
 array([[  9.47246348, 116.45972223],
        [123.19132688,   2.42921781]]))

This gives a constraint file like
```python
 {
    'type': 'brick',	
    'remove_asa_shell': False,   # True or False

    # lower left front corner of the orthorhombic box
    'chain1': 'B',              
    'res_num1': 22,             
    'atom1': 'NH2',             
    'offset1': (0,0, -5),
                                
    # upper right back corner
    'chain2': 'F',              
    'res_num2': 128,            
    'atom2': 'O',              
    'offset2': (0,0,10),    # extend towards IC
 }   
```